In [17]:
import pandas as pd
import numpy as np
import os
import sys
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [18]:
def load_settings(path='CSV/settings_oos.csv'):
    df = pd.read_csv(path, delimiter=';', index_col='Parameter')
    def get(key, cast=str):
        return cast(str(df.loc[key, 'Value']).strip())
    stg = {
        'is_scenario_name': get('is_scenario_name'),
        'oos_scenario_name': get('oos_scenario_name'),
        'z_window': get('z_window', int),
        'commission_pct': get('commission_pct', float),
        'initial_capital': get('initial_capital', float),
        'positioning_mode': get('positioning_mode'),  # 'beta' или 'dollar'
        'max_open_pairs': get('max_open_pairs', float),
        'z_entry': get('z_entry', float),
        'z_exit': get('z_exit', float),
        'hl_multiplier': get('hl_multiplier', float),
        'min_hedge_ratio_pct': get('min_hedge_ratio_pct', float),
    }
    return stg

STG = load_settings()

print("="*65)
print(f"{' НАСТРОЙКИ OOS-ДВИЖКА ':-^65}")
print(f"  > IS-сценарий (отбор пар):   {STG['is_scenario_name']}")
print(f"  > OOS-сценарий (торговля):   {STG['oos_scenario_name']}")
print(f"  > Z-window:                  {STG['z_window']} дней")
print(f"  > Positioning:               {STG['positioning_mode']}-neutral")
print(f"  > z_entry / z_exit:          {STG['z_entry']} / {STG['z_exit']}")
print(f"  > HL multiplier:             {STG['hl_multiplier']}")
print(f"  > Max open pairs:            {int(STG['max_open_pairs'])}")
print(f"  > Min hedge ratio:           {STG['min_hedge_ratio_pct']*100:.0f}% "
      f"(защита от вырожденного beta)")
print(f"  > Commission per leg:        {STG['commission_pct']*100:.3f}%")
print(f"  > Initial capital:           {STG['initial_capital']:,.0f}")
print("="*65)

--------------------- НАСТРОЙКИ OOS-ДВИЖКА ----------------------
  > IS-сценарий (отбор пар):   PURE
  > OOS-сценарий (торговля):   PURE
  > Z-window:                  30 дней
  > Positioning:               beta-neutral
  > z_entry / z_exit:          1.5 / 0.75
  > HL multiplier:             4.0
  > Max open pairs:            15
  > Min hedge ratio:           10% (защита от вырожденного beta)
  > Commission per leg:        0.000%
  > Initial capital:           100,000


In [19]:
print("Загрузка цен...")
PRICES = pd.read_excel('DATA/FULL_Trading_Calendar.xlsx', index_col=0, parse_dates=True)
print(f"Загружено: {PRICES.shape[1]} тикеров, {len(PRICES)} дней "
      f"({PRICES.index.min().date()} - {PRICES.index.max().date()})")

Загрузка цен...
Загружено: 525 тикеров, 2901 дней (2015-01-02 - 2026-07-17)


In [20]:
IS_PATH = f"DATA/Output/{STG['is_scenario_name']}/IS_{STG['is_scenario_name']}.xlsx"
print(f"Загрузка IS-результатов: {IS_PATH}")
is_df = pd.read_excel(IS_PATH, sheet_name='Results')

sched = pd.read_csv('CSV/schedule.csv', sep=';')
sched['Iteration'] = range(1, len(sched) + 1)
sched['OOS_Start'] = pd.to_datetime(sched['OOS_Start'], format='%d.%m.%Y')
sched['OOS_End'] = pd.to_datetime(sched['OOS_End'], format='%d.%m.%Y')

is_df = is_df.merge(sched[['Iteration', 'OOS_Start', 'OOS_End']], on='Iteration', how='left')
is_df = is_df.sort_values(['OOS_Start', 'Iteration']).reset_index(drop=True)

print(f"Загружено пар: {len(is_df)} по {is_df['Iteration'].nunique()} итерациям")
print(f"OOS диапазон: {is_df['OOS_Start'].min().date()} - {is_df['OOS_Start'].max().date()}")

Загрузка IS-результатов: DATA/Output/PURE/IS_PURE.xlsx
Загружено пар: 88977 по 108 итерациям
OOS диапазон: 2017-01-01 - 2025-12-01


In [21]:
def calculate_mdd(equity_series):
    """Максимальная просадка (%) по ряду equity."""
    if len(equity_series) < 1:
        return 0.0
    peaks = np.maximum.accumulate(equity_series)
    drawdowns = (peaks - equity_series) / peaks
    return drawdowns.max() * 100


def get_rolling_z(history, spread_now, window, min_periods):
    """Rolling Z-score: последние window точек истории + текущий спред."""
    hist = (history + [spread_now])[-window:]
    if len(hist) < min_periods:
        return np.nan
    arr = np.array(hist, dtype=float)
    mu, sigma = arr.mean(), arr.std()
    if sigma == 0:
        return np.nan
    return (spread_now - mu) / sigma


def prewarm_spread_history(t1, t2, beta, intercept, before_date, window, all_prices):
    """Прогрев rolling-окна историей ДО before_date (без заглядывания вперёд)."""
    if t1 not in all_prices.columns or t2 not in all_prices.columns:
        return []
    pre_idx = all_prices.index[all_prices.index < before_date]
    if len(pre_idx) == 0:
        return []
    pre_slice = pre_idx[-window:]
    hist = []
    for dt in pre_slice:
        p1v = all_prices.loc[dt, t1]
        p2v = all_prices.loc[dt, t2]
        if not pd.isna(p1v) and not pd.isna(p2v):
            hist.append(float(p1v) - beta * float(p2v) - intercept)
    return hist[-window:]


def size_position(limit_per_pair, p1, p2, beta, mode):
    """Beta-neutral (по умолчанию) или dollar-neutral (для ablation-теста)."""
    if mode == 'dollar':
        qty1 = (limit_per_pair / 2) / p1
        qty2 = (limit_per_pair / 2) / p2
    else:
        qty1 = limit_per_pair / (p1 + abs(beta) * p2)
        qty2 = qty1 * abs(beta)
    return qty1, qty2

In [22]:
z_window = STG['z_window']
min_per = max(z_window // 2, 10)
z_entry = STG['z_entry']
z_exit = STG['z_exit']
hl_mult = STG['hl_multiplier']
max_open = STG['max_open_pairs']
commission_pct = STG['commission_pct']
initial_capital = STG['initial_capital']
positioning_mode = STG['positioning_mode']
min_hedge_ratio_pct = STG['min_hedge_ratio_pct']

start_date = is_df['OOS_Start'].min()
oos_end_date = is_df['OOS_Start'].max() + pd.offsets.MonthEnd(1)
end_date = PRICES.index[-1]

all_tickers = list(set(is_df['Asset_A'].tolist() + is_df['Asset_B'].tolist()))
valid_tickers = [t for t in all_tickers if t in PRICES.columns]
prices = PRICES.loc[start_date:end_date, valid_tickers].ffill()

print(f"OOS период:      {start_date.date()} \u2014 {oos_end_date.date()}")
print(f"Trailing период: {oos_end_date.date()} \u2014 {end_date.date()}")
print(f"Rolling Z окно:  {z_window} дней (min для входа: {min_per})")

spread_histories = {}
warmed_pairs = {}
_cached_month_start = None
curr_month_params = {}

active_trades = {}
banned_pairs = {}
closed_trades = []
daily_equity = []
daily_active_counts = []
all_days = prices.index

for i, current_date in enumerate(all_days):
    in_trailing = current_date > oos_end_date
    curr_month_start = current_date.replace(day=1)

    # --- пересборка параметров месяца + прогрев новых пар ---
    if curr_month_start != _cached_month_start:
        _cached_month_start = curr_month_start
        curr_month_params = {}
        for _, r in is_df[is_df['OOS_Start'] == curr_month_start].iterrows():
            pk = f"{r['Asset_A']}-{r['Asset_B']}"
            curr_month_params[pk] = {
                'beta': r['Beta'], 'intercept': r['Intercept'],
                't1': r['Asset_A'], 't2': r['Asset_B'],
                'half_life': r['Half_Life'], 'type_a': r['Type_A'], 'type_b': r['Type_B'],
            }
        for pk, m in curr_month_params.items():
            if pk in active_trades:
                continue
            param_key = (round(m['beta'], 6), round(m['intercept'], 6))
            if pk not in warmed_pairs or warmed_pairs[pk] != param_key:
                spread_histories[pk] = prewarm_spread_history(
                    m['t1'], m['t2'], m['beta'], m['intercept'],
                    current_date, z_window, PRICES
                )
                warmed_pairs[pk] = param_key

    eligible_keys = set(curr_month_params.keys())
    tracked_keys = eligible_keys | set(active_trades.keys()) | set(banned_pairs.keys())

    # --- обновление истории спредов ---
    for pk in tracked_keys:
        if pk in active_trades:
            tr = active_trades[pk]
            beta_u, int_u, t1, t2 = tr['Beta'], tr['Intercept'], tr['Ticker_1'], tr['Ticker_2']
        elif pk in banned_pairs:
            bp = banned_pairs[pk]
            beta_u, int_u, t1, t2 = bp['beta'], bp['intercept'], bp['ticker_1'], bp['ticker_2']
        elif pk in curr_month_params:
            m = curr_month_params[pk]
            beta_u, int_u, t1, t2 = m['beta'], m['intercept'], m['t1'], m['t2']
        else:
            continue
        if t1 not in prices.columns or t2 not in prices.columns:
            continue
        p1v, p2v = prices.loc[current_date, t1], prices.loc[current_date, t2]
        if pd.isna(p1v) or pd.isna(p2v):
            continue
        sp = float(p1v) - beta_u * float(p2v) - int_u
        hist = spread_histories.get(pk, [])
        hist.append(sp)
        if len(hist) > z_window:
            hist = hist[-z_window:]
        spread_histories[pk] = hist

    # --- equity ---
    unrealized_pnl = 0
    for pk, trade in active_trades.items():
        if trade['Ticker_1'] not in prices.columns or trade['Ticker_2'] not in prices.columns:
            continue
        p1 = prices.loc[current_date, trade['Ticker_1']]
        p2 = prices.loc[current_date, trade['Ticker_2']]
        if pd.isna(p1) or pd.isna(p2):
            continue
        if trade['Spread_Type'] == "Long Spread":
            unrealized_pnl += (p1 - trade['Price_In_1']) * trade['Qty_1'] + \
                               (trade['Price_In_2'] - p2) * trade['Qty_2']
        else:
            unrealized_pnl += (trade['Price_In_1'] - p1) * trade['Qty_1'] + \
                               (p2 - trade['Price_In_2']) * trade['Qty_2']

    realized_pnl = sum(t.get('Total_PnL_USD', 0) for t in closed_trades)
    current_equity = initial_capital + realized_pnl + unrealized_pnl
    daily_equity.append(current_equity)
    daily_active_counts.append(len(active_trades))

    mode_tag = "[TRAIL]" if in_trailing else "[OOS]  "
    sys.stdout.write(f"\r{mode_tag} {current_date.date()} | Открыто: {len(active_trades)} | "
                      f"Бан: {len(banned_pairs)} | Equity: {current_equity:.2f}")
    sys.stdout.flush()

    # --- снятие бана ---
    to_unban = []
    for pk, bp in banned_pairs.items():
        t1, t2 = bp['ticker_1'], bp['ticker_2']
        if t1 not in prices.columns or t2 not in prices.columns:
            to_unban.append(pk); continue
        p1b, p2b = prices.loc[current_date, t1], prices.loc[current_date, t2]
        if pd.isna(p1b) or pd.isna(p2b):
            continue
        sp_now = float(p1b) - bp['beta'] * float(p2b) - bp['intercept']
        hist = spread_histories.get(pk, [])
        zb = get_rolling_z(hist[:-1], sp_now, z_window, min_periods=5)
        if not pd.isna(zb) and abs(zb) <= z_exit:
            to_unban.append(pk)
    for pk in to_unban:
        banned_pairs.pop(pk, None)

    # --- выход (конвергенция или тайм-аут) ---
    to_close = []
    for pair_key, trade in active_trades.items():
        s1, s2 = trade['Ticker_1'], trade['Ticker_2']
        if s1 not in prices.columns or s2 not in prices.columns:
            to_close.append(pair_key); continue
        p1, p2 = prices.loc[current_date, s1], prices.loc[current_date, s2]
        if pd.isna(p1) or pd.isna(p2):
            continue

        sp_now = float(p1) - trade['Beta'] * float(p2) - trade['Intercept']
        hist = spread_histories.get(pair_key, [])
        z = get_rolling_z(hist[:-1], sp_now, z_window, min_periods=5)
        if pd.isna(z):
            continue

        is_conv = (trade['Spread_Type'] == "Long Spread" and z >= -z_exit) or \
                  (trade['Spread_Type'] == "Short Spread" and z <= z_exit)
        duration = (current_date - pd.to_datetime(trade['Entry_Date'])).days
        is_time_out = duration > (trade.get('Half_Life', 10) * hl_mult)

        if is_conv or is_time_out:
            q1, q2 = trade['Qty_1'], trade['Qty_2']
            if trade['Spread_Type'] == "Long Spread":
                trade_pnl = (p1 - trade['Price_In_1']) * q1 + (trade['Price_In_2'] - p2) * q2
            else:
                trade_pnl = (trade['Price_In_1'] - p1) * q1 + (p2 - trade['Price_In_2']) * q2

            commission = commission_pct * (
                trade['Price_In_1'] * q1 + trade['Price_In_2'] * q2 + p1 * q1 + p2 * q2
            )
            trade_pnl -= commission
            reason = "Zero_Cross" if is_conv else "Time_Exit"

            if is_time_out and not is_conv:
                banned_pairs[pair_key] = {
                    'beta': trade['Beta'], 'intercept': trade['Intercept'],
                    'ticker_1': s1, 'ticker_2': s2
                }

            closed_trades.append({
                **trade,
                "Year": current_date.year,
                "Price_Out_1": p1, "Price_Out_2": p2,
                "Commission_USD": round(commission, 4),
                "Total_PnL_USD": round(trade_pnl, 2),
                "Total_PnL_Pct": round((trade_pnl / initial_capital) * 100, 4),
                "Exit_Date": current_date.date(), "Exit_Reason": reason,
                "Exit_Z_Score": round(z, 2), "Duration_Days": duration,
                "Trailing": in_trailing
            })
            to_close.append(pair_key)

    for pk in to_close:
        active_trades.pop(pk, None)

    # --- вход (только в пределах OOS, не в trailing) ---
    if not in_trailing:
        iter_pairs = is_df[is_df['OOS_Start'] == curr_month_start]
        if not iter_pairs.empty and len(active_trades) < max_open:
            # приоритет слотам — по силе корреляции (сильнейшая статистическая связь первой)
            iter_pairs = iter_pairs.reindex(
                iter_pairs['Pearson_Corr'].abs().sort_values(ascending=False).index
            )
            limit_per_pair = current_equity / max_open
            for _, row in iter_pairs.iterrows():
                if len(active_trades) >= max_open:
                    break
                pair_key = f"{row['Asset_A']}-{row['Asset_B']}"
                if pair_key in active_trades or pair_key in banned_pairs:
                    continue
                if row['Asset_A'] not in prices.columns or row['Asset_B'] not in prices.columns:
                    continue

                p1, p2 = prices.loc[current_date, row['Asset_A']], prices.loc[current_date, row['Asset_B']]
                if pd.isna(p1) or pd.isna(p2):
                    continue

                sp_now = float(p1) - row['Beta'] * float(p2) - row['Intercept']
                hist = spread_histories.get(pair_key, [])
                z = get_rolling_z(hist[:-1], sp_now, z_window, min_per)
                if pd.isna(z):
                    continue

                if abs(z) > z_entry:
                    beta = row['Beta']
                    t_type = "Short Spread" if z > z_entry else "Long Spread"
                    qty1, qty2 = size_position(limit_per_pair, p1, p2, beta, positioning_mode)

                    # --- защита от вырожденного хеджа (см. methodology_check.md) ---
                    # При экстремальной beta (напр. DOGE~BTC, цены отличаются на порядки)
                    # вторая нога может схлопнуться почти в ноль -> фактически голая позиция.
                    notional1 = qty1 * p1
                    notional2 = qty2 * p2
                    if notional1 <= 0 or notional2 <= 0:
                        continue
                    hedge_ratio = min(notional1, notional2) / max(notional1, notional2)
                    if hedge_ratio < min_hedge_ratio_pct:
                        continue

                    active_trades[pair_key] = {
                        "Iteration": row['Iteration'], "Pair": pair_key,
                        "Type_A": row['Type_A'], "Type_B": row['Type_B'],
                        "Entry_Date": current_date.date(), "Spread_Type": t_type,
                        "Ticker_1": row['Asset_A'], "Price_In_1": p1, "Qty_1": qty1,
                        "Ticker_2": row['Asset_B'], "Price_In_2": p2, "Qty_2": qty2,
                        "Beta": beta, "Intercept": row['Intercept'],
                        "Half_Life": row.get('Half_Life', 10), "Year": current_date.year
                    }

    # --- trailing: если всё закрыто, останавливаемся ---
    if in_trailing and len(active_trades) == 0:
        print(f"\n[INFO] Все позиции закрыты {current_date.date()}, trailing завершён.")
        all_days = all_days[:i + 1]
        daily_equity = daily_equity[:i + 1]
        daily_active_counts = daily_active_counts[:i + 1]
        break

print("\nЦикл завершён.")

OOS период:      2017-01-01 — 2025-12-31
Trailing период: 2025-12-31 — 2026-07-17
Rolling Z окно:  30 дней (min для входа: 15)
[TRAIL] 2026-01-27 | Открыто: 1 | Бан: 1 | Equity: 18937.8700
[INFO] Все позиции закрыты 2026-01-27, trailing завершён.

Цикл завершён.


In [23]:
if not closed_trades:
    print("Закрытых сделок не было.")
    df_t = pd.DataFrame()
    df_open = pd.DataFrame()
else:
    df_t = pd.DataFrame(closed_trades)

    yearly = df_t.groupby('Year').agg(
        PnL_USD=('Total_PnL_USD', 'sum'), PnL_Pct=('Total_PnL_Pct', 'sum'),
        Trades=('Pair', 'count'),
    ).reset_index()

    yearly['Yearly_PF'] = 0.0
    for idx, r_y in yearly.iterrows():
        yr_d = df_t[df_t['Year'] == r_y['Year']]
        gp = yr_d[yr_d['Total_PnL_USD'] > 0]['Total_PnL_USD'].sum()
        gl = abs(yr_d[yr_d['Total_PnL_USD'] < 0]['Total_PnL_USD'].sum())
        yearly.at[idx, 'Yearly_PF'] = round(gp / gl if gl != 0 else float('inf'), 2)

    equity_series = pd.Series(daily_equity, index=all_days[:len(daily_equity)])
    daily_returns = equity_series.pct_change().dropna()
    strat_sharpe = (daily_returns.mean() / daily_returns.std() * np.sqrt(252)) if daily_returns.std() != 0 else 0
    downside_std = daily_returns[daily_returns < 0].std()
    strat_sortino = (daily_returns.mean() / downside_std * np.sqrt(252)) if downside_std != 0 else 0

    yearly_sharpe_map = daily_returns.groupby(daily_returns.index.year).apply(
        lambda x: (x.mean() / x.std() * np.sqrt(252)) if x.std() != 0 else 0)
    yearly_mdd_map = equity_series.groupby(equity_series.index.year).apply(calculate_mdd)

    yearly = yearly.merge(yearly_mdd_map.rename('MDD_Pct'), left_on='Year', right_index=True)
    yearly = yearly.merge(yearly_sharpe_map.rename('Yearly_Sharpe'), left_on='Year', right_index=True)

    real_util = (sum(daily_active_counts) / len(daily_active_counts) / max_open) * 100
    total_mdd = calculate_mdd(equity_series)
    total_ret = ((daily_equity[-1] / initial_capital) - 1) * 100
    days_total = (equity_series.index[-1] - equity_series.index[0]).days
    years_total = days_total / 365.25
    cagr = ((daily_equity[-1] / initial_capital) ** (1 / years_total) - 1) * 100 if years_total > 0 else 0
    gp_total = df_t[df_t['Total_PnL_USD'] > 0]['Total_PnL_USD'].sum()
    gl_total = abs(df_t[df_t['Total_PnL_USD'] < 0]['Total_PnL_USD'].sum())
    profit_factor = round(gp_total / gl_total if gl_total != 0 else float('inf'), 2)
    total_commission = df_t['Commission_USD'].sum()
    trailing_count = int(df_t.get('Trailing', pd.Series(False)).sum())

    # --- открытые позиции на конец периода ---
    last_date = all_days[-1]
    open_positions = []
    for pair_key, trade in active_trades.items():
        s1, s2 = trade['Ticker_1'], trade['Ticker_2']
        p1 = prices.loc[last_date, s1] if s1 in prices.columns else np.nan
        p2 = prices.loc[last_date, s2] if s2 in prices.columns else np.nan
        q1, q2 = trade['Qty_1'], trade['Qty_2']
        if not pd.isna(p1) and not pd.isna(p2):
            if trade['Spread_Type'] == "Long Spread":
                unreal = (p1 - trade['Price_In_1']) * q1 + (trade['Price_In_2'] - p2) * q2
            else:
                unreal = (trade['Price_In_1'] - p1) * q1 + (p2 - trade['Price_In_2']) * q2
        else:
            unreal = np.nan
        duration = (last_date - pd.to_datetime(trade['Entry_Date'])).days
        open_positions.append({
            **trade, "Last_Price_1": p1, "Last_Price_2": p2,
            "Duration_Days": duration, "Unrealized_PnL_USD": unreal
        })
    df_open = pd.DataFrame(open_positions) if open_positions else pd.DataFrame()
    open_unrealized = df_open['Unrealized_PnL_USD'].sum() if not df_open.empty else 0.0

    summary_data = pd.DataFrame({
        "Metric": ["Total Return (%)", "Annual Return CAGR (%)", "Max Drawdown (%)", "Profit Factor",
                   "Win Rate (%)", "Sharpe Ratio", "Sortino Ratio", "Avg Duration (days)",
                   "Capital Utilization (%)", "Total Commission (USD)", "Closed in Trailing",
                   "Open Positions (EOD)", "Unrealized PnL (USD)", "Z Window (days)",
                   "Positioning Mode", "IS Scenario", "OOS Scenario"],
        "Value": [round(total_ret, 3), round(cagr, 2), round(total_mdd, 2), profit_factor,
                  round((df_t['Total_PnL_USD'] > 0).mean() * 100, 2), round(strat_sharpe, 2),
                  round(strat_sortino, 2), round(df_t['Duration_Days'].mean(), 1), round(real_util, 1),
                  round(total_commission, 2), trailing_count, len(df_open), round(open_unrealized, 2),
                  z_window, positioning_mode, STG['is_scenario_name'], STG['oos_scenario_name']]
    })

    ex_an = df_t.groupby('Exit_Reason').agg(
        Trades=('Pair', 'count'), Avg_PnL_Pct=('Total_PnL_Pct', 'mean')
    ).reset_index()

    print("\n" + "=" * 65)
    print(f"{' ИТОГОВАЯ СТАТИСТИКА (OOS + TRAILING) ':-^65}")
    print(summary_data.to_string(index=False))
    print("\n[ ГОДОВАЯ ДИНАМИКА ]")
    print(yearly.to_string(index=False))
    print("\n[ АНАЛИЗ ВЫХОДОВ ]")
    print(ex_an.to_string(index=False))


------------- ИТОГОВАЯ СТАТИСТИКА (OOS + TRAILING) --------------
                 Metric   Value
       Total Return (%) -81.062
 Annual Return CAGR (%)  -16.77
       Max Drawdown (%)   86.43
          Profit Factor    0.71
           Win Rate (%)   66.96
           Sharpe Ratio   -0.32
          Sortino Ratio   -0.33
    Avg Duration (days)    13.9
Capital Utilization (%)    99.5
 Total Commission (USD)     0.0
     Closed in Trailing      15
   Open Positions (EOD)       0
   Unrealized PnL (USD)     0.0
        Z Window (days)      30
       Positioning Mode    beta
            IS Scenario    PURE
           OOS Scenario    PURE

[ ГОДОВАЯ ДИНАМИКА ]
 Year   PnL_USD  PnL_Pct  Trades  Yearly_PF   MDD_Pct  Yearly_Sharpe
 2017   7217.50   7.2182     394       1.35  7.616865       0.827176
 2018  -4913.87  -4.9133     367       0.84 14.105136      -0.427303
 2019  -3195.79  -3.1957     381       0.88 10.372560      -0.392392
 2020  -1897.35  -1.8979     438       0.97 24.216843      

In [24]:
out_dir = f"DATA/Output/{STG['is_scenario_name']}/{z_window}"
os.makedirs(out_dir, exist_ok=True)
out_file = f"{out_dir}/OOS_{STG['oos_scenario_name']}.xlsx"

if closed_trades:
    with pd.ExcelWriter(out_file, engine='openpyxl') as writer:
        df_t.to_excel(writer, sheet_name='OOS_Trades', index=False)
        yearly.to_excel(writer, sheet_name='Yearly_Stats', index=False)
        summary_data.to_excel(writer, sheet_name='Summary', index=False)
        ex_an.to_excel(writer, sheet_name='Exit_Analysis', index=False)
        if not df_open.empty:
            df_open.to_excel(writer, sheet_name='Open_Positions', index=False)
    print(f"\n[DONE] Результаты сохранены: {out_file}")
else:
    print("\n[!] Нет закрытых сделок \u2014 файл не сохранён.")


[DONE] Результаты сохранены: DATA/Output/PURE/30/OOS_PURE.xlsx
